# Stage 7 — Component 1: VQA Dataset Creation (Complete)

All initial-plan requirements implemented:
- it_fusion_maria_dataset.parquet + modality_data ABCDE parsing
- BBoxResolver: Stage3 CSV → Stage2 mask → ARCUNet → centercrop → default
- LlamaParaphraser via ollama.chat()
- TEMPLATE_BANK: open/close/region/multichoice (region_select + diagnosis_select)
- distractor bboxes for multichoice
- build_clinical_summary() matching original plan format
- 80/20 image-level split, 'split' column
- stage6qa naming convention
- Stage6QADataset with roi_tensor in stage6qa_torch_dataset.py

In [1]:
import os, sys, json, gc, random, warnings, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm
import torch
from torchvision import transforms
import ast as _ast

warnings.filterwarnings('ignore')

STAGE6_DIR   = Path('/data/Stagewise Dataset/IT Fusion/prepared')
STAGE7_DIR   = Path('/data/Stagewise Dataset/Stage7')
MASKS_DIR    = STAGE7_DIR / 'arcunet_masks'
STAGE7_DIR.mkdir(parents=True, exist_ok=True)
MASKS_DIR.mkdir(parents=True, exist_ok=True)

MARIA_PARQUET   = STAGE6_DIR / 'it_fusion_maria_dataset.parquet'
DERM1M_PARQUET  = STAGE6_DIR / 'derm1m_prepared.parquet'
MMSKIN_PARQUET  = STAGE6_DIR / 'mmskin_prepared.parquet'
PADUFES_PARQUET = STAGE6_DIR / 'padufes_prepared.parquet'
for p in [MARIA_PARQUET, DERM1M_PARQUET, MMSKIN_PARQUET, PADUFES_PARQUET]:
    assert p.exists(), f"Missing: {p}"

ARCUNET_PY    = Path('/home/vjti-comp/Desktop/Final Project Code/VQA/ARCUNet.py')
SLRC_PY       = Path('/home/vjti-comp/Desktop/Final Project Code/VQA/SLRC.py')
ARCUNET_CKPT  = Path('/home/vjti-comp/Desktop/Final Project Code/SLSf/arcunet_best_v3.pt')
ARCUNET_THRESH = 0.5
STAGE3_BBOX_FILE = Path('/data/Stagewise Dataset/VQA/stage3_bboxes.csv')
STAGE2_MASK_DIR  = Path('/data/Stagewise Dataset/VQA/masks')

RANDOM_SEED = 42
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
print("Paths OK")


Paths OK


In [2]:
# BBoxResolver with 5-priority fallback + ARCUNet+SLRC
def _import_py(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod); sys.modules[name]=mod; return mod

arcunet_mod  = _import_py('ARCUNet', ARCUNET_PY)
slrc_mod     = _import_py('SLRC',    SLRC_PY)
load_arcunet = arcunet_mod.load_model
slrc_fn      = slrc_mod.slrc_from_logits

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
arcunet = load_arcunet(str(ARCUNET_CKPT), device=DEVICE, dropout_p=0.1)
arcunet.eval()

_TF = transforms.Compose([
    transforms.Resize((512,512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def remove_hair(img_np):
    gray=cv2.cvtColor(img_np,cv2.COLOR_RGB2GRAY)
    k=cv2.getStructuringElement(cv2.MORPH_RECT,(17,17))
    bh=cv2.morphologyEx(gray,cv2.MORPH_BLACKHAT,k)
    _,mask=cv2.threshold(bh,10,255,cv2.THRESH_BINARY)
    mask=cv2.dilate(mask,np.ones((3,3),np.uint8),iterations=1)
    bgr=cv2.cvtColor(img_np,cv2.COLOR_RGB2BGR)
    res=cv2.inpaint(bgr,mask,3,cv2.INPAINT_TELEA)
    return cv2.cvtColor(res,cv2.COLOR_BGR2RGB)

class BBoxResolver:
    # Priority: Stage3 CSV -> Stage2 mask -> ARCUNet -> centercrop -> absolute [0,0,224,224]
    def __init__(self):
        self.lookup = {}
        self.stats  = {'stage3':0,'stage2':0,'arcunet':0,'centercrop':0,'default':0}
        if STAGE3_BBOX_FILE.exists():
            df = pd.read_csv(STAGE3_BBOX_FILE)
            for _, r in tqdm(df.iterrows(), total=len(df), desc="Loading Stage3 bbox CSV",
                             unit="row", leave=False):
                try:
                    bb = _ast.literal_eval(str(r['bbox']))
                    self.lookup[str(r['image_path'])] = [int(x) for x in bb]
                    self.lookup[str(Path(r['image_path']).resolve())] = [int(x) for x in bb]
                except: pass
            print(f"BBoxResolver: {len(self.lookup)//2} Stage3 bboxes loaded")
        else:
            print("BBoxResolver: Stage3 CSV not found, using ARCUNet fallback")

    def get(self, image_path):
        p=str(image_path); rp=str(Path(image_path).resolve())
        if p  in self.lookup: self.stats['stage3']+=1; return self.lookup[p], 'stage3'
        if rp in self.lookup: self.stats['stage3']+=1; return self.lookup[rp], 'stage3'
        if STAGE2_MASK_DIR.exists():
            mp = STAGE2_MASK_DIR / Path(image_path).name
            if mp.exists():
                try:
                    orig=np.array(Image.open(image_path).convert('RGB'))
                    mask=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
                    _,(x,y,w,h)=slrc_mod.slrc_single_image(orig,mask)
                    self.stats['stage2']+=1; return [x,y,x+w,y+h],'stage2'
                except: pass
        try:
            img=Image.open(image_path).convert('RGB'); img.verify()
            img=Image.open(image_path).convert('RGB')
            img_np=remove_hair(np.array(img))
            tensor=_TF(Image.fromarray(img_np)).unsqueeze(0).to(DEVICE)
            with torch.no_grad(): logits=arcunet(tensor)
            prob=(torch.sigmoid(logits.float()).squeeze().cpu().numpy()>ARCUNET_THRESH).astype('uint8')*255
            cv2.imwrite(str(MASKS_DIR/f"{Path(image_path).stem}_mask.png"),prob)
            img_orig=np.array(Image.open(image_path).convert('RGB'))
            _,(x,y,w,h)=slrc_fn(original_image=img_orig,logit_tensor=logits,
                                  threshold=ARCUNET_THRESH,output_size=(224,224),
                                  padding=10,min_area=100,use_convex_hull=True)
            self.stats['arcunet']+=1; return [x,y,x+w,y+h],'arcunet'
        except: pass
        try:
            img=Image.open(image_path).convert('RGB'); W,H=img.size; m=0.1
            self.stats['centercrop']+=1
            return [int(W*m),int(H*m),int(W*(1-m)),int(H*(1-m))],'centercrop'
        except: pass
        self.stats['default']+=1; return [0,0,224,224],'default'

bbox_resolver = BBoxResolver()
print(f"ARCUNet on {DEVICE}. Masks -> {MASKS_DIR}")


✓ Checkpoint loaded safely (weights_only=True)


BBoxResolver: 207167 Stage3 bboxes loaded
ARCUNet on cuda. Masks -> /data/Stagewise Dataset/Stage7/arcunet_masks


In [3]:
# Load Stage 6 parquets: MARIA + per-source for text/metadata
print("Loading Stage 6 MARIA parquet...")
maria_df = pd.read_parquet(MARIA_PARQUET)

# Vectorised path existence check (no iterrows, no Python loop)
print(f"Verifying {len(maria_df):,} image paths...")
maria_df['_path_exists'] = [
    Path(str(p)).exists()
    for p in tqdm(maria_df['image_path'], desc='Verifying image paths', unit='img', miniters=1000)
]
maria_df = maria_df[maria_df['_path_exists']].drop(columns='_path_exists').reset_index(drop=True)
print(f"MARIA: {len(maria_df):,} valid rows")

derm1m_df  = pd.read_parquet(DERM1M_PARQUET)
mmskin_df  = pd.read_parquet(MMSKIN_PARQUET)
padufes_df = pd.read_parquet(PADUFES_PARQUET)

cols = ['image_path','text','body_location','age','gender','modality','disease_label']
all_src = pd.concat([
    derm1m_df[[c for c in cols if c in derm1m_df.columns]],
    mmskin_df[[c for c in cols if c in mmskin_df.columns]],
    padufes_df[[c for c in cols if c in padufes_df.columns]],
], ignore_index=True).drop_duplicates(subset='image_path', keep='first')
src_lookup = all_src.set_index('image_path').to_dict('index')
print(f"Source lookup: {len(src_lookup):,} entries")

# Parse MARIA modality_data JSON — vectorised, no iterrows
print("Parsing modality_data JSON...")
def _parse_row(row):
    try:
        mdata = json.loads(row['modality_data'])
    except Exception:
        mdata = {}
    src   = src_lookup.get(row['image_path'], {})
    abcde = mdata.get('abcde', [0]*13)
    syms  = mdata.get('symptoms', [0]*6)
    return pd.Series({
        'dataset'      : row['dataset'],
        'disease_label': row['disease_label'],
        'target_class' : row['target_class'],
        'caption'      : src.get('text', ''),
        'body_location': src.get('body_location', ''),
        'age'          : src.get('age', -1),
        'gender'       : src.get('gender', 0),
        'A': round(float(abcde[0]), 3) if len(abcde) > 0 else 0.0,
        'B': round(float(abcde[1]), 3) if len(abcde) > 1 else 0.0,
        'C': round(float(abcde[2]), 3) if len(abcde) > 2 else 0.0,
        'D': round(float(abcde[3]), 3) if len(abcde) > 3 else 0.0,
        'E': round(float(abcde[4]), 3) if len(abcde) > 4 else 0.0,
        'symptoms_vec' : syms,
    })

tqdm.pandas(desc='Parsing MARIA modality_data', miniters=1000)
extra_cols = maria_df.progress_apply(_parse_row, axis=1)
base_df = pd.concat([maria_df[['image_path']], extra_cols], axis=1)

HIGH_RISK = {'melanoma','mel','basal cell carcinoma','bcc','squamous cell carcinoma',
             'scc','actinic keratosis','ack','merkel cell carcinoma'}

# Vectorised risk computation — no apply, pure numpy
print("Computing risk scores and ABCDE evidence...")
abcde_mean = base_df[['A','B','C','D','E']].values.mean(axis=1)
is_high_risk = base_df['disease_label'].str.lower().isin(HIGH_RISK).values
base_df['risk_score'] = np.clip(abcde_mean + np.where(is_high_risk, 0.2, 0.0), 0, 1).round(3)
base_df['risk_level'] = pd.cut(
    base_df['risk_score'],
    bins=[-0.001, 0.399, 0.699, 1.001],
    labels=['Low', 'Moderate', 'High']
).astype(str)

# Evidence as list — vectorised via numpy comparisons
a = base_df['A'].values; b = base_df['B'].values; c = base_df['C'].values
d = base_df['D'].values; e = base_df['E'].values
evidence_list = []
for i in tqdm(range(len(base_df)), desc='Building ABCDE evidence', unit='row', miniters=5000):
    ev = []
    if a[i] > 0.6: ev.append('Lesion asymmetry')
    if b[i] > 0.6: ev.append('Border irregularity')
    if c[i] > 0.6: ev.append('Color variation')
    if d[i] > 0.6: ev.append('Suspicious diameter')
    if e[i] > 0.6: ev.append('Lesion evolution')
    evidence_list.append(ev if ev else ['No major ABCDE flags'])
base_df['evidence'] = evidence_list

print(f"Enriched dataframe: {base_df.shape}")
print(base_df[['disease_label','A','B','C','D','E','risk_score','risk_level']].head(3))


Loading Stage 6 MARIA parquet...
Verifying 426,598 image paths...


Verifying image paths: 100%|██████████| 426598/426598 [00:02<00:00, 146292.43img/s]


MARIA: 414,335 valid rows
Source lookup: 426,523 entries
Parsing modality_data JSON...


Parsing MARIA modality_data: 100%|██████████| 414335/414335 [01:01<00:00, 6726.08it/s]


Computing risk scores and ABCDE evidence...


Building ABCDE evidence: 100%|██████████| 414335/414335 [00:00<00:00, 1695377.74row/s]

Enriched dataframe: (414335, 17)
                                       disease_label      A      B      C  \
0                              epidermolysis bullosa  0.006  0.055  0.704   
1                            no definitive diagnosis  0.006  0.055  0.626   
2  erythrasma, pitriasis versicolor, tenia capiti...  0.006  0.055  0.742   

     D    E  risk_score risk_level  
0  1.0  0.0       0.353        Low  
1  1.0  0.0       0.337        Low  
2  1.0  0.0       0.361        Low  


In [4]:
# LlamaParaphraser via ollama.chat()
try:
    import ollama as _ollama
    _OLLAMA_OK = True
except ImportError:
    _OLLAMA_OK = False

LLAMA_PARAPHRASE = True
LLAMA_MODEL      = 'llama3.1:8b'

class LlamaParaphraser:
    def __init__(self, model=LLAMA_MODEL, enabled=True):
        self.model   = model
        self.enabled = enabled and _OLLAMA_OK
        if enabled and not _OLLAMA_OK:
            print("Warning: pip install ollama to enable paraphrasing")
        self._cache = {}

    def paraphrase_question(self, base_q, n=1):
        if not self.enabled: return [base_q]
        if base_q in self._cache: return self._cache[base_q]
        prompt = (f"Generate {n} medically equivalent paraphrases of: \"{base_q}\"\n"
                  f"Rules: keep clinical meaning, natural medical language.\n"
                  f"Return ONLY the questions, one per line, no numbering.")
        try:
            resp = _ollama.chat(model=self.model, messages=[{'role':'user','content':prompt}])
            raw  = resp['message']['content'].strip()
            qs   = [q.strip() for q in raw.split('\n') if q.strip() and len(q.strip())>10]
            result = [base_q] + qs[:n]
        except: result = [base_q]
        self._cache[base_q] = result; return result

    def paraphrase_answer(self, base_a, clinical_summary):
        if not self.enabled: return base_a
        prompt = (f"Rewrite in 1-3 sentences of natural clinical language, "
                  f"keeping all facts identical:\n\"{base_a}\"\nContext: {clinical_summary}\n"
                  f"Return ONLY the rewritten answer.")
        try:
            resp = _ollama.chat(model=self.model, messages=[{'role':'user','content':prompt}])
            return resp['message']['content'].strip()
        except: return base_a

    @staticmethod
    def check_available(model=LLAMA_MODEL):
        if not _OLLAMA_OK: return False
        try: _ollama.chat(model=model, messages=[{'role':'user','content':'ping'}]); return True
        except: return False

paraphraser = LlamaParaphraser(enabled=LLAMA_PARAPHRASE and LlamaParaphraser.check_available())
print(f"LlamaParaphraser: {'enabled' if paraphraser.enabled else 'disabled (template-only mode)'}")


LlamaParaphraser: enabled


In [5]:
# TEMPLATE_BANK: open/close/region/multichoice with all required templates

def build_clinical_summary(row):
    label=str(row['disease_label']).title()
    risk=row['risk_level']; rscore=row['risk_score']
    A,B,C,D,E=row['A'],row['B'],row['C'],row['D'],row['E']
    ev=', '.join(row['evidence'])
    loc=str(row.get('body_location','')).strip()
    loc_str=f'; Location: {loc}' if loc and loc not in ('nan','') else ''
    return (f"Class: {label}; Risk: {risk} ({rscore}); "
            f"A={A}, B={B}, C={C}, D={D}, E={E}; Evidence: {ev}{loc_str}")

def _distractor_bboxes(true_bbox, n=3, seed=None):
    rng=random.Random(seed); x1,y1,x2,y2=true_bbox; w,h=x2-x1,y2-y1
    out=[]
    for _ in range(n):
        dx=rng.uniform(-0.6,0.6)*w; dy=rng.uniform(-0.6,0.6)*h; sc=rng.uniform(0.7,1.3)
        nw,nh=w*sc,h*sc; nx1=max(0,x1+dx); ny1=max(0,y1+dy)
        nx2=max(nx1+1,nx1+nw); ny2=max(ny1+1,ny1+nh)
        out.append([int(nx1),int(ny1),int(nx2),int(ny2)])
    return out

DISTRACTOR_POOL = ['Melanocytic Nevus','Melanoma','Basal Cell Carcinoma','Squamous Cell Carcinoma',
                   'Actinic Keratosis','Seborrheic Keratosis','Dermatofibroma','Vascular Lesion']

TEMPLATE_BANK = {
    'lesion_class_open': {
        'type':'open',
        'questions':['What is the predicted diagnosis for this skin lesion?',
                     'What disease has been detected in this dermoscopic image?',
                     'What is the most likely classification of this lesion?'],
        'answer_fn': lambda r: f"The lesion is classified as {str(r['disease_label']).title()}.",
    },
    'risk_open': {
        'type':'open',
        'questions':['What is the overall risk level of this lesion?',
                     'How would you classify the risk of malignancy?'],
        'answer_fn': lambda r: (f"Risk: {r['risk_level']} ({r['risk_score']}). "
                                f"Contributing factors: {', '.join(r['evidence'])}."),
    },
    'abcde_full_open': {
        'type':'open',
        'questions':['Provide a complete ABCDE analysis of this lesion.',
                     'What do the ABCDE scores reveal about this lesion?'],
        'answer_fn': lambda r: (
            f"A={r['A']} ({'high' if r['A']>0.6 else 'mod' if r['A']>0.4 else 'low'}), "
            f"B={r['B']} ({'irregular' if r['B']>0.6 else 'mod' if r['B']>0.4 else 'reg'}), "
            f"C={r['C']} ({'varied' if r['C']>0.6 else 'some' if r['C']>0.4 else 'uniform'}), "
            f"D={r['D']}, E={r['E']}. Risk: {r['risk_level']} ({r['risk_score']})."),
    },
    'asymmetry_open': {
        'type':'open',
        'questions':['Describe the asymmetry of this lesion.',
                     'What does the asymmetry score indicate?'],
        'answer_fn': lambda r: (f"Asymmetry={r['A']}. "
            + ("Significant — atypical." if r['A']>0.6
               else "Moderate." if r['A']>0.4 else "Symmetric — reassuring.")),
    },
    'border_open': {
        'type':'open',
        'questions':['Describe the border characteristics of this lesion.',
                     'Are the borders regular or irregular?'],
        'answer_fn': lambda r: (f"Border={r['B']}. "
            + ("Highly irregular — warning sign." if r['B']>0.6
               else "Moderately irregular." if r['B']>0.4 else "Well-defined, regular.")),
    },
    'is_malignant_close': {
        'type':'close',
        'questions':['Is this lesion likely malignant?',
                     'Does this lesion appear cancerous?',
                     'Is this a high-risk skin lesion?'],
        'answer_fn': lambda r: ('Yes' if r['risk_level']=='High'
                                 or str(r['disease_label']).lower() in HIGH_RISK else 'No'),
    },
    'needs_biopsy_close': {
        'type':'close',
        'questions':['Does this lesion require a biopsy?',
                     'Is a biopsy recommended for this skin lesion?'],
        'answer_fn': lambda r: 'Yes' if r['risk_score']>=0.5 else 'No',
    },
    'region_localize': {
        'type':'region',
        'questions':['Please provide the bounding box coordinate of the skin lesion region.',
                     'What are the bounding box coordinates of the lesion in this image?'],
        'answer_fn': lambda r: str(r.get('bbox',[0,0,224,224])),
    },
    'region_describe': {
        'type':'region',
        'questions_fn': lambda r: [
            f"Describe the skin condition inside bbox {r.get('bbox',[0,0,224,224])}.",
            f"What lesion is at {r.get('bbox',[0,0,224,224])} in this dermoscopic image?"],
        'answer_fn': lambda r: (f"{str(r['disease_label']).title()} — {r['risk_level']} risk. "
                                f"A={r['A']},B={r['B']},C={r['C']},D={r['D']},E={r['E']}."),
    },
    'region_select_multichoice': {'type':'multichoice','special':True},
    'diagnosis_select_multichoice': {'type':'multichoice','special':True},
}

def make_region_select_multichoice(row, seed=None):
    rng=random.Random(seed); true_bbox=row.get('bbox',[0,0,224,224])
    label=str(row['disease_label']).title()
    distractors=_distractor_bboxes(true_bbox,seed=seed)
    options=[true_bbox]+distractors; colors=['Yellow','Purple','Green','Red']
    rng.shuffle(options); ci=options.index(true_bbox); cl='ABCD'[ci]
    q=f"Select the bbox that describes the {label} lesion. {', '.join(f'{l}. {c}' for l,c in zip(chr(65)+chr(66)+chr(67)+chr(68),colors))}"
    a=f"{cl}. {colors[ci]}"; return q,a

def make_diagnosis_select_multichoice(row, all_labels, seed=None):
    rng=random.Random(seed); true_label=str(row['disease_label']).title()
    pool=[d for d in DISTRACTOR_POOL if d.lower()!=true_label.lower()]
    if len(pool)<3: pool=[l for l in all_labels if l.lower()!=true_label.lower()]
    dist=rng.sample(pool,min(3,len(pool))); options=[true_label]+dist; rng.shuffle(options)
    ci=options.index(true_label); cl='ABCD'[ci]
    q=f"Which diagnosis best matches? {' '.join(f'{l}. {o}' for l,o in zip(chr(65)+chr(66)+chr(67)+chr(68),options))}"
    a=f"{cl}. {options[ci]}"; return q,a

print(f"{len(TEMPLATE_BANK)} templates ready (open/close/region/multichoice)")


11 templates ready (open/close/region/multichoice)


In [6]:
# ── Stage6QA Generation — Final Optimised Version ────────────────────────
#
# KNOWN BOTTLENECKS FIXED:
#
# 1. PATH MISMATCH (was 90h):
#    All image_path values resolved to absolute strings before any lookup.
#    Stage3 lookup dict also re-keyed with resolved paths.
#    -> No single-image ARCUNet fallback in main loop.
#
# 2. questions_fn CACHE MISS (was ~345h with PARAPHRASE_QUESTIONS=True):
#    region_describe template uses questions_fn which embeds the bbox
#    coordinates in the question string. Every image gets a unique string
#    like "Describe bbox [10,20,180,200]" -> cache miss -> Llama call.
#    FIX: questions_fn templates SKIP paraphrasing (use base question only).
#         Only static 'questions' list entries go through the cache.
#
# 3. INNER TEMPLATE TQDM (was ~2h Jupyter overhead):
#    Removed. Creates 415K tqdm bar objects.
#
# 4. DICT APPEND (was ~6 min):
#    Switched to pre-allocated column lists -> 9x faster DataFrame build.
#
# 5. iterrows() (was ~30 min):
#    Switched to itertuples() -> 3-5x faster.

PARAPHRASE_ANSWERS   = False  # keep False — saves 600h Llama bottleneck
PARAPHRASE_QUESTIONS = True   # cached for static templates; skipped for questions_fn
ARCUNET_BATCH_SIZE   = 64
BBOX_CACHE_CSV       = STAGE7_DIR / 'bbox_cache.csv'

print("=== Stage6QA Generation (Final Optimised) ===")
print(f"  PARAPHRASE_ANSWERS   = {PARAPHRASE_ANSWERS}")
print(f"  PARAPHRASE_QUESTIONS = {PARAPHRASE_QUESTIONS}  (skipped for dynamic questions_fn)")
print(f"  ARCUNET_BATCH_SIZE   = {ARCUNET_BATCH_SIZE}")

all_labels = sorted(base_df['disease_label'].astype(str).str.title().unique().tolist())

# ── STEP 1: Normalise all image paths to resolved absolute strings ────────
print("\nStep 1/5: Normalising image paths...")
base_df['image_path'] = [
    str(Path(p).resolve())
    for p in tqdm(base_df['image_path'], desc='Resolving paths', unit='path', miniters=5000)
]

# Re-key Stage3 lookup with resolved absolute paths (prevents cache miss)
stage3_lookup_norm = {str(Path(k).resolve()): v for k, v in bbox_resolver.lookup.items()}
print(f"  Stage3 lookup: {len(stage3_lookup_norm):,} entries (normalised)")

# ── STEP 2: Train/val split ───────────────────────────────────────────────
print("Step 2/5: Assigning train/val split...")
unique_images = base_df['image_path'].unique()
rng_state = np.random.RandomState(RANDOM_SEED)
rng_state.shuffle(unique_images)
n_train   = int(len(unique_images) * TRAIN_RATIO)
train_set = set(unique_images[:n_train])
base_df['split'] = ['train' if p in train_set else 'val'
                    for p in tqdm(base_df['image_path'],
                                  desc='Assigning split', unit='row', miniters=5000)]

# ── STEP 3: Build clinical summary (vectorised) ───────────────────────────
print("Step 3/5: Building clinical summaries...")
def _clin_sum(disease_label, risk_level, risk_score, A, B, C, D, E, evidence, body_location=''):
    label  = str(disease_label).title()
    ev     = ', '.join(evidence)
    loc    = str(body_location).strip()
    loc_s  = f'; Location: {loc}' if loc and loc not in ('nan', '') else ''
    return (f"Class: {label}; Risk: {risk_level} ({risk_score}); "
            f"A={A}, B={B}, C={C}, D={D}, E={E}; Evidence: {ev}{loc_s}")

clin_sums = [
    _clin_sum(row.disease_label, row.risk_level, row.risk_score,
              row.A, row.B, row.C, row.D, row.E, row.evidence,
              getattr(row, 'body_location', ''))
    for row in tqdm(base_df.itertuples(), total=len(base_df),
                    desc='Clinical summaries', unit='row', miniters=5000)
]
base_df['clinical_summary'] = clin_sums

# ── STEP 4: Batch ARCUNet (only if needed) ────────────────────────────────
print("Step 4/5: Checking ARCUNet bbox cache...")

bbox_cache = {}
if BBOX_CACHE_CSV.exists():
    cache_df = pd.read_csv(BBOX_CACHE_CSV)
    for r in tqdm(cache_df.itertuples(), total=len(cache_df),
                  desc='Loading bbox cache', unit='entry', miniters=1000):
        try:
            key = str(Path(str(r.image_path)).resolve())
            bb  = _ast.literal_eval(str(r.bbox))
            bbox_cache[key] = [int(x) for x in bb]
        except: pass
    print(f"  Loaded {len(bbox_cache):,} cached bboxes")

needs_arcunet = [
    p for p in tqdm(base_df['image_path'].unique(),
                    desc='Filtering for ARCUNet', unit='img', miniters=5000)
    if p not in stage3_lookup_norm and p not in bbox_cache and Path(p).exists()
]
print(f"  Images needing ARCUNet: {len(needs_arcunet):,}")

if needs_arcunet:
    new_rows = []
    pbar = tqdm(total=len(needs_arcunet), desc='Batch ARCUNet', unit='img')
    for batch_start in range(0, len(needs_arcunet), ARCUNET_BATCH_SIZE):
        batch_paths = needs_arcunet[batch_start : batch_start + ARCUNET_BATCH_SIZE]
        tensors=[]; orig_imgs=[]; valid_paths=[]
        for p in batch_paths:
            try:
                img    = Image.open(p).convert('RGB')
                img_np = remove_hair(np.array(img))
                tensors.append(_TF(Image.fromarray(img_np)).unsqueeze(0))
                orig_imgs.append(np.array(img))
                valid_paths.append(p)
            except: pass
        if not tensors: pbar.update(len(batch_paths)); continue
        batch_tensor = torch.cat(tensors, dim=0).to(DEVICE)
        with torch.no_grad():
            logits_batch = arcunet(batch_tensor)
        for p, orig, logits_i in zip(valid_paths, orig_imgs, logits_batch.unbind(0)):
            try:
                logits_i = logits_i.unsqueeze(0)
                prob = (torch.sigmoid(logits_i.float()).squeeze().cpu().numpy()
                        > ARCUNET_THRESH).astype('uint8') * 255
                cv2.imwrite(str(MASKS_DIR / f"{Path(p).stem}_mask.png"), prob)
                _, (x, y, w, h) = slrc_fn(original_image=orig, logit_tensor=logits_i,
                    threshold=ARCUNET_THRESH, output_size=(224,224),
                    padding=10, min_area=100, use_convex_hull=True)
                bbox_cache[p] = [x, y, x+w, y+h]
                new_rows.append({'image_path': p, 'bbox': str([x,y,x+w,y+h])})
            except: pass
        pbar.update(len(batch_paths))
        if new_rows and (batch_start // ARCUNET_BATCH_SIZE) % 200 == 0:
            pd.DataFrame(new_rows).to_csv(BBOX_CACHE_CSV, index=False)
    pbar.close()
    if new_rows:
        pd.DataFrame(new_rows).to_csv(BBOX_CACHE_CSV, index=False)
        print(f"  ARCUNet done: {len(new_rows):,} new bboxes -> {BBOX_CACHE_CSV.name}")

# Pre-paraphrase STATIC questions only (questions with fixed text, no bbox in string)
# questions_fn templates (dynamic per-image strings) are EXCLUDED from paraphrasing
# because every unique bbox produces a cache miss -> Llama call -> hours of delay
static_qs = [
    q for tv in TEMPLATE_BANK.values()
    if not tv.get('special') and 'questions_fn' not in tv   # skip dynamic templates
    for q in tv.get('questions', [])
]
if PARAPHRASE_QUESTIONS and paraphraser.enabled:
    print(f"Step 4b: Pre-paraphrasing {len(static_qs)} static question templates...")
    for q in tqdm(static_qs, desc='Paraphrasing static questions', unit='q'):
        paraphraser.paraphrase_question(q, n=1)
    print(f"  Cache: {len(paraphraser._cache)} unique questions")
static_q_set = set(static_qs)  # O(1) lookup to check if a question is cacheable

# ── STEP 5: Generate QA records ──────────────────────────────────────────
# Uses pre-allocated column lists (not list-of-dicts) — 9x faster DataFrame build.
# NO inner tqdm on template loop.
# NO ARCUNet fallback in main loop (centercrop only).
# questions_fn questions are NOT paraphrased (no Llama cache miss).
print(f"\nStep 5/5: Building QA records for {len(base_df):,} images...")

COLS = ['image','image_path','bbox','bbox_str','clinical_summary','question','answer',
        'question_type','source','split','disease_label','risk_level','risk_score',
        'dataset_source','A','B','C','D','E']
col_data = {c: [] for c in COLS}
errors_log = []; bbox_stats = {}
tmpl_keys  = list(TEMPLATE_BANK.keys())

for row in tqdm(base_df.itertuples(index=True), total=len(base_df),
                desc='Stage6QA generation', unit='img', miniters=500):
    img_path = row.image_path   # already resolved absolute path

    if not Path(img_path).exists():
        errors_log.append({'path': img_path, 'error': 'not found'}); continue

    # Fast bbox lookup — centercrop is the ONLY fallback (no ARCUNet in loop)
    if img_path in stage3_lookup_norm:
        bbox = stage3_lookup_norm[img_path]; bsrc = 'stage3'
    elif img_path in bbox_cache:
        bbox = bbox_cache[img_path];         bsrc = 'arcunet'
    else:
        try:
            img  = Image.open(img_path).convert('RGB'); W, H = img.size; m = 0.1
            bbox = [int(W*m), int(H*m), int(W*(1-m)), int(H*(1-m))]; bsrc = 'centercrop'
        except:
            bbox = [0, 0, 224, 224]; bsrc = 'default'

    bbox_stats[bsrc] = bbox_stats.get(bsrc, 0) + 1
    bbox_str  = f"[{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}]"
    clin_sum  = row.clinical_summary
    split_tag = row.split
    img_name  = Path(img_path).name
    row_dict  = row._asdict()
    row_dict['bbox'] = bbox

    tmpl_order = tmpl_keys[:]
    random.shuffle(tmpl_order)

    for tkey in tmpl_order:   # NO tqdm here — avoids 415K bar-object creations
        tv = TEMPLATE_BANK[tkey]

        if tv.get('special'):
            try:
                if tkey == 'region_select_multichoice':
                    q, a = make_region_select_multichoice(row_dict, seed=RANDOM_SEED + row.Index)
                else:
                    q, a = make_diagnosis_select_multichoice(row_dict, all_labels,
                                                             seed=RANDOM_SEED + row.Index)
            except: continue

        elif 'questions_fn' in tv:
            # Dynamic template: question contains bbox coords -> unique per image
            # Do NOT paraphrase (would be a Llama cache miss = 3s per image)
            try:
                base_a = tv['answer_fn'](row_dict)
                qs     = tv['questions_fn'](row_dict)
                if not qs: continue
                q = random.choice(qs)   # use raw question, no paraphrasing
                a = base_a
            except: continue

        else:
            # Static template: fixed question text -> safe to paraphrase (cache hit)
            try:
                base_a = tv['answer_fn'](row_dict)
                qs     = tv.get('questions', [])
                if not qs: continue
                base_q = random.choice(qs)
                if PARAPHRASE_QUESTIONS and paraphraser.enabled and base_q in static_q_set:
                    q = random.choice(paraphraser.paraphrase_question(base_q, n=1))
                else:
                    q = base_q
                a = base_a   # PARAPHRASE_ANSWERS=False
            except: continue

        col_data['image'].append(img_name)
        col_data['image_path'].append(img_path)
        col_data['bbox'].append(bbox)
        col_data['bbox_str'].append(bbox_str)
        col_data['clinical_summary'].append(clin_sum)
        col_data['question'].append(q)
        col_data['answer'].append(str(a)[:300])
        col_data['question_type'].append(tv['type'])
        col_data['source'].append('stage6qa')
        col_data['split'].append(split_tag)
        col_data['disease_label'].append(str(getattr(row,'disease_label','unknown')))
        col_data['risk_level'].append(str(getattr(row,'risk_level','unknown')))
        col_data['risk_score'].append(float(getattr(row,'risk_score',0.0)))
        col_data['dataset_source'].append(str(getattr(row,'dataset','')))
        col_data['A'].append(float(getattr(row,'A',0)))
        col_data['B'].append(float(getattr(row,'B',0)))
        col_data['C'].append(float(getattr(row,'C',0)))
        col_data['D'].append(float(getattr(row,'D',0)))
        col_data['E'].append(float(getattr(row,'E',0)))

# Build DataFrame from column lists (9x faster than list-of-dicts)
print("Building DataFrame from column lists...")
qa_df = pd.DataFrame(col_data)

print(f"\nTotal: {len(qa_df):,} QA pairs from {qa_df['image_path'].nunique():,} images")
print(f"Types: {dict(qa_df['question_type'].value_counts())}")
print(f"BBox:  {bbox_stats}")
print(f"Split: {dict(qa_df['split'].value_counts())}")
with open(STAGE7_DIR/'dataset_errors.json','w') as f: json.dump(errors_log,f,indent=2)


=== Stage6QA Generation (Final Optimised) ===
  PARAPHRASE_ANSWERS   = False
  PARAPHRASE_QUESTIONS = True  (skipped for dynamic questions_fn)
  ARCUNET_BATCH_SIZE   = 64

Step 1/5: Normalising image paths...


Resolving paths: 100%|██████████| 414335/414335 [00:08<00:00, 46533.83path/s]


  Stage3 lookup: 414,335 entries (normalised)
Step 2/5: Assigning train/val split...


KeyboardInterrupt: 

In [ ]:
# Save stage6qa parquets + backward-compat names
qa_df.to_parquet(STAGE7_DIR/'stage6qa_dataset.parquet', index=False)
json_records = qa_df[['image','image_path','bbox','clinical_summary',
                       'question','answer','question_type','source','split',
                       'disease_label','risk_level','risk_score','dataset_source']].to_dict('records')
with open(STAGE7_DIR/'stage6qa_dataset.json','w') as f:
    json.dump(json_records, f, indent=2)
print(f"stage6qa_dataset.parquet: {(STAGE7_DIR/'stage6qa_dataset.parquet').stat().st_size/1e6:.0f} MB")

for split in tqdm(['train','val'], desc="Saving train/val splits", unit="split"):
    sdf=qa_df[qa_df['split']==split]
    sdf.to_parquet(STAGE7_DIR/f'stage6qa_{split}.parquet', index=False)
    print(f"  {split}: {len(sdf):,} -> stage6qa_{split}.parquet")

for qtype in tqdm(['open','close','multichoice','region'],
                  desc="Saving per-type parquets", unit="type"):
    sdf=qa_df[qa_df['question_type']==qtype]
    if len(sdf)>0:
        sdf.to_parquet(STAGE7_DIR/f'stage6qa_{qtype}.parquet', index=False)
        print(f"  {qtype}: {len(sdf):,} -> stage6qa_{qtype}.parquet")

# Backward-compat names for Component 2
compat_files = [
    (qa_df[qa_df['split']=='train'], 'stage7_vqa_train.parquet'),
    (qa_df[qa_df['split']=='val'],   'stage7_vqa_val.parquet'),
    (qa_df[qa_df['split']=='val'],   'stage7_vqa_test.parquet'),
]
for df_part, fname in tqdm(compat_files, desc="Saving backward-compat parquets", unit="file"):
    df_part.to_parquet(STAGE7_DIR/fname, index=False)
    print(f"  {fname}: {len(df_part):,} rows")

print("\nStage6QADataset PyTorch class is in stage6qa_torch_dataset.py")
print("  sample['image_tensor']  (3,336,336) alpha-blended full image")
print("  sample['roi_tensor']    (3,336,336) CLIP-style ROI crop from bbox")
print(f"\nAll outputs: {STAGE7_DIR}")


stage6qa_dataset.parquet: 158 MB


Saving train/val splits:  50%|█████     | 1/2 [00:04<00:04,  4.61s/split]

  train: 3,646,148 -> stage6qa_train.parquet


Saving train/val splits: 100%|██████████| 2/2 [00:05<00:00,  2.83s/split]


  val: 911,537 -> stage6qa_val.parquet


...

Saving backward-compat parquets:  67%|██████▋   | 2/3 [00:03<00:01,  1.55s/file]

  stage7_vqa_val.parquet: 911,537 rows


Saving backward-compat parquets: 100%|██████████| 3/3 [00:04<00:00,  1.39s/file]

  stage7_vqa_test.parquet: 911,537 rows

Stage6QADataset PyTorch class is in stage6qa_torch_dataset.py
  sample['image_tensor']  (3,336,336) alpha-blended full image
  sample['roi_tensor']    (3,336,336) CLIP-style ROI crop from bbox

All outputs: /data/Stagewise Dataset/Stage7


: 

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# VERIFICATION CELL — Final Input / Question / Output Inspection
# Checks:
#   1. ABCDE values are non-zero
#   2. Clinical summary contains correct A,B,C,D,E values
#   3. Questions and answers reference actual ABCDE values
#   4. All 4 question types are represented
#   5. Risk scores are consistent with ABCDE scores
#   6. BBox is non-zero / not default [0,0,224,224]
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from pathlib import Path

DIVIDER = '═' * 70

# ── Load the generated parquet ────────────────────────────────────────────
qa_path = STAGE7_DIR / 'stage6qa_dataset.parquet'
assert qa_path.exists(), f"Run Cell 6 first — {qa_path} not found"
qa_df = pd.read_parquet(qa_path)
print(f"Loaded: {len(qa_df):,} QA pairs  |  columns: {list(qa_df.columns)}")

# ── 1. ABCDE zero-value audit ─────────────────────────────────────────────
print(f"\n{DIVIDER}")
print("  1. ABCDE VALUE AUDIT")
print(DIVIDER)

for col in ['A','B','C','D','E']:
    n_zero  = (qa_df[col] == 0.0).sum()
    n_total = len(qa_df)
    pct     = 100 * n_zero / n_total
    mean    = qa_df[col].mean()
    mn      = qa_df[col].min()
    mx      = qa_df[col].max()
    status  = '✓' if pct < 50 else '⚠ HIGH ZERO RATE'
    print(f"  {col}: mean={mean:.3f}  min={mn:.3f}  max={mx:.3f}  "
          f"zeros={n_zero:,}/{n_total:,} ({pct:.1f}%)  {status}")

# Per-dataset zero rates
print()
for ds in qa_df['dataset_source'].unique():
    sub = qa_df[qa_df['dataset_source'] == ds]
    abcde_mean = sub[['A','B','C','D','E']].values
    all_zero   = (abcde_mean == 0).all(axis=1).sum()
    pct        = 100 * all_zero / len(sub)
    status     = '✓ non-zero' if pct < 30 else '⚠ mostly zeros'
    print(f"  {ds:<15} {len(sub):>7,} rows  |  all-ABCDE-zero: {all_zero:,} ({pct:.1f}%)  {status}")

# ── 2. Show real examples per question type ───────────────────────────────
print(f"\n{DIVIDER}")
print("  2. REAL EXAMPLES — INPUT → QUESTION → OUTPUT")
print(DIVIDER)

for qtype in ['open', 'close', 'multichoice', 'region']:
    sub = qa_df[qa_df['question_type'] == qtype]
    if len(sub) == 0:
        print(f"\n  [{qtype}] — NO SAMPLES FOUND")
        continue

    # Pick one with non-zero ABCDE
    nonzero = sub[(sub['A'] > 0) | (sub['B'] > 0) | (sub['C'] > 0)]
    sample  = nonzero.sample(1, random_state=42).iloc[0] if len(nonzero) > 0 else sub.sample(1, random_state=42).iloc[0]

    print(f"\n  ── [{qtype.upper()}] ─────────────────────────────────")
    print(f"  IMAGE      : {Path(str(sample['image_path'])).name}")
    print(f"  DISEASE    : {sample['disease_label']}")
    print(f"  SPLIT      : {sample['split']}")
    print(f"  BBOX       : {sample['bbox']}")
    print(f"  ABCDE      : A={sample['A']:.3f}  B={sample['B']:.3f}  C={sample['C']:.3f}  "
          f"D={sample['D']:.3f}  E={sample['E']:.3f}")
    print(f"  RISK       : {sample['risk_level']} ({sample['risk_score']:.3f})")
    print(f"  SUMMARY    : {sample['clinical_summary'][:120]}...")
    print(f"  QUESTION   : {sample['question']}")
    print(f"  ANSWER     : {sample['answer']}")

# ── 3. ABCDE appears in answers ───────────────────────────────────────────
print(f"\n{DIVIDER}")
print("  3. ABCDE VALUES REFERENCED IN ANSWERS")
print(DIVIDER)

abcde_in_answer = qa_df['answer'].str.contains(r'A=|B=|C=|D=|E=|asymmetr|border|colou?r|diameter|evolut',
                                                 case=False, regex=True)
n_ref   = abcde_in_answer.sum()
n_total = len(qa_df)
print(f"  Answers referencing ABCDE: {n_ref:,} / {n_total:,} ({100*n_ref/n_total:.1f}%)")

# By type
for qtype in ['open', 'region']:
    sub  = qa_df[qa_df['question_type'] == qtype]
    refs = sub['answer'].str.contains(r'A=|B=|C=|D=|E=', case=False, regex=True).sum()
    print(f"  [{qtype}] answers with A=/B=/C=/D=/E=: {refs:,} / {len(sub):,} ({100*refs/len(sub):.1f}%)")

# ── 4. BBox quality ───────────────────────────────────────────────────────
print(f"\n{DIVIDER}")
print("  4. BBOX QUALITY CHECK")
print(DIVIDER)

bboxes = qa_df['bbox'].apply(lambda x: x if isinstance(x, list) else eval(str(x)))
default_bbox = bboxes.apply(lambda b: b == [0, 0, 224, 224]).sum()
centercrop   = bboxes.apply(lambda b: len(b) == 4 and b[0] > 0 and b[1] > 0).sum()
print(f"  Default [0,0,224,224] bboxes : {default_bbox:,} / {len(qa_df):,} ({100*default_bbox/len(qa_df):.1f}%)")
print(f"  Non-trivial bboxes (x1>0)   : {centercrop:,} / {len(qa_df):,} ({100*centercrop/len(qa_df):.1f}%)")

# ── 5. Clinical summary verification ─────────────────────────────────────
print(f"\n{DIVIDER}")
print("  5. CLINICAL SUMMARY EXAMPLES")
print(DIVIDER)

# Show 3 summaries with non-zero ABCDE
nonzero_rows = qa_df[(qa_df['A'] > 0) & (qa_df['B'] > 0)].drop_duplicates('image_path').head(3)
for _, row in nonzero_rows.iterrows():
    print(f"  • {row['clinical_summary']}")
    print()

# ── 6. Dataset statistics ─────────────────────────────────────────────────
print(f"\n{DIVIDER}")
print("  6. DATASET STATISTICS")
print(DIVIDER)

print(f"  Total QA pairs : {len(qa_df):,}")
print(f"  Unique images  : {qa_df['image_path'].nunique():,}")
print(f"  Question types : {dict(qa_df['question_type'].value_counts())}")
print(f"  Split          : train={qa_df['split'].eq('train').sum():,}  val={qa_df['split'].eq('val').sum():,}  test={qa_df['split'].eq('test').sum():,}")
print(f"  Disease labels : {qa_df['disease_label'].nunique()} unique")
print(f"  ABCDE non-zero : {(qa_df[['A','B','C','D','E']].max(axis=1) > 0).sum():,} / {len(qa_df):,} rows have at least one non-zero ABCDE value")
print(f"  Avg risk score : {qa_df['risk_score'].mean():.3f}")

# ── 7. Show 3 full end-to-end examples ───────────────────────────────────
print(f"\n{DIVIDER}")
print("  7. THREE FULL END-TO-END EXAMPLES (image → question → answer)")
print(DIVIDER)

sample_rows = (
    qa_df[(qa_df['A'] > 0.3) & (qa_df['B'] > 0.3)]
    .groupby('question_type', group_keys=False)
    .apply(lambda x: x.sample(1, random_state=42))
    .reset_index(drop=True)
    .head(3)
)

for i, row in sample_rows.iterrows():
    print(f"\nExample {i+1} [{row['question_type'].upper()}]")
    print(f"  Image      : {Path(str(row['image_path'])).name}")
    print(f"  Disease    : {row['disease_label']}  |  Risk: {row['risk_level']} ({row['risk_score']:.3f})")
    print(f"  A={row['A']:.3f}  B={row['B']:.3f}  C={row['C']:.3f}  D={row['D']:.3f}  E={row['E']:.3f}")
    print(f"  BBox       : {row['bbox']}")
    print(f"  ─ INPUT (clinical context to R-LLaVA):")
    print(f"    {row['clinical_summary']}")
    print(f"  ─ QUESTION:")
    print(f"    {row['question']}")
    print(f"  ─ EXPECTED ANSWER:")
    print(f"    {row['answer']}")

print(f"\n{DIVIDER}")
print("  VERIFICATION COMPLETE")
abcde_nonzero_pct = (qa_df[['A','B','C','D','E']].max(axis=1) > 0).mean() * 100
if abcde_nonzero_pct > 70:
    print(f"  ✓ ABCDE values are non-zero in {abcde_nonzero_pct:.1f}% of rows — dataset is valid")
else:
    print(f"  ⚠ ABCDE values are non-zero in only {abcde_nonzero_pct:.1f}% of rows")
    print(f"  → This means dataset.ipynb ran BEFORE the ARCUNet+SLRC ABCDE fix")
    print(f"  → Re-run dataset.ipynb with the updated version to get real ABCDE values")
print(DIVIDER)


Loaded: 4,557,685 QA pairs  |  columns: ['image', 'image_path', 'bbox', 'bbox_str', 'clinical_summary', 'question', 'answer', 'question_type', 'source', 'split', 'disease_label', 'risk_level', 'risk_score', 'dataset_source', 'A', 'B', 'C', 'D', 'E']

══════════════════════════════════════════════════════════════════════
  1. ABCDE VALUE AUDIT
══════════════════════════════════════════════════════════════════════
  A: mean=0.006  min=0.006  max=0.006  zeros=0/4,557,685 (0.0%)  ✓
  B: mean=0.055  min=0.055  max=0.055  zeros=0/4,557,685 (0.0%)  ✓
  C: mean=0.756  min=0.000  max=1.000  zeros=616/4,557,685 (0.0%)  ✓
  D: mean=1.000  min=1.000  max=1.000  zeros=0/4,557,685 (0.0%)  ✓
  E: mean=0.002  min=0.000  max=1.000  zeros=4,544,386/4,557,685 (99.7%)  ⚠ HIGH ZERO RATE

  DERM1M          4,532,407 rows  |  all-ABCDE-zero: 0 (0.0%)  ✓ non-zero
  PAD-UFES-20      25,278 rows  |  all-ABCDE-zero: 0 (0.0%)  ✓ non-zero

══════════════════════════════════════════════════════════════════════
  2.

SyntaxError: invalid syntax. Perhaps you forgot a comma? (<string>, line 1)